In [ ]:
# Run this in a Jupyter terminal or !pip cell
!pip install keplergl geopandas

In [ ]:
import pandas
import pyarrow
import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

try:
    SparkContext.getOrCreate().stop()
except:
    pass

In [ ]:
# 1. Setup Spark
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("NYC-Taxi-Ingestion") \
    .getOrCreate()

# Print the Spark version to verify successful connection
print(f"Spark version: {spark.version}")

In [ ]:
spark.sparkContext.master

In [ ]:
output_dir = "/home/jovyan/work/data"
file_path = f"{output_dir}/fhvhv_Jan2025.parquet"

## Load parquet file and understand its contents

In [ ]:
df = spark.read.parquet(file_path)

In [ ]:
df.show(5)

In [ ]:
print(f"Total Records: {df.count():,}")

# 2. Number of columns and their names
print(f"Number of Columns: {len(df.columns)}")
print(f"Column Names: {df.columns}")

In [ ]:
df.printSchema()

## Visualizing data 

In [ ]:
from pyspark.sql import functions as F
import pandas as pd

# ── 1. Stratified sample — 500K rows from 20M (~2.5%)
#    fraction chosen to land near 500K; adjust if needed
SAMPLE_FRACTION = 0.025

df_sample = df.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=42)
print(f"Sampled rows: {df_sample.count():,}")

# ── 2. Load your Excel zone file into Spark
zone_pd = pd.read_csv(f"{output_dir}/taxi_zone_lookup.csv")   
zone_spark = spark.createDataFrame(zone_pd)

# ── 3. Join zone names onto sample (PU side)
df_sample = df_sample.join(
    zone_spark.select(
        F.col("LocationID").alias("PULocationID"),
        F.col("Zone").alias("PU_Zone"),
        F.col("Borough").alias("PU_Borough")
    ), on="PULocationID", how="left"
).join(
    zone_spark.select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("Zone").alias("DO_Zone"),
        F.col("Borough").alias("DO_Borough")
    ), on="DOLocationID", how="left"
)

# ── 4. Select only what Kepler needs — keep it lean
df_map = df_sample.select(
    "PULocationID", "DOLocationID",
    "PU_Zone", "PU_Borough",
    "DO_Zone", "DO_Borough",
    "base_passenger_fare", "tips", "trip_miles"
)

map_pd = df_map.toPandas()
print(f"Pandas shape: {map_pd.shape}")